# LLM - Klasyczne metody klasyfikacji tekstu - LAB

# Zadanie

Zaadaptuj kod z notatnika *LLM - Klasyczne metody klasyfikacji tekstu - Omówienie* do problemu klasyfikacji liczby gwiazdek dla opinii z serwisu Yelp.
Możesz przygotować pętlę treningową albo w czystym PyTorchu, albo z wykorzystaniem biblioteki PyTorch Lightning.

* Wykorzystaj zbiór `Yelp/yelp_review_full` ([link](https://huggingface.co/datasets/Yelp/yelp_review_full)) zawierający opinie z serwisu Yelp (kolumna: `text`) i etykietę (kolumna: `label`) o wartościach $0,1,2,3,4$ określającą liczbę gwiazdek przyznaną przez użytkownika (a ściślej, liczbę gwiazdek minus jeden). Ponieważ mamy pięć klas, ostatnia warstwa liniowa w sieci neuronowej musi zwracać pięć wartości.
    * Zgodnie z dobrą praktyką z części treningowej wydziel dodatkową część walidacyjną i testową.
    * Ogranicz rozmiar każdej części zbioru danych (treningowej, walidacyjnej i testowej). Część treningowa nie powinna zawierać więcej niż 100k elementów.
* Do ekstrakcji cech z tekstu wykorzystaj **metodę TF-IDF** (*term frequency-inverse document frequency*) opartą o podejście typu worek słów (*bag-of-words*). Zastosuj funkcję `TfidfVectorizer` z biblioteki `scikit-learn`.
* Można wykorzystać podobną architekturę sieci (perceptron wielowarstwowy z warstwą Dropout) jak w notatniku wykładowym.



## Punkty do wykonania

1.   Napisz funkcję znajdującą i wyświetlającą $k$ elementów zbioru testowego dla których model najbardziej się myli, czyli estymuje najmniejsze prawdopodobieństwa prawdziwej klasy. Softmax jest funkcją ściśle rosnącą, więc wystarczy znaleźć elementy z najmniejszą wartością nieznormalizowanego wyjścia z sieci (logita) dla prawdziwej klasy.
2.   Zbadaj wpływ wybranych parametrów funkcji ekstrakcji cech z tekstu `TfidfVectorizer` na skuteczność wytrenowanego modelu. Uruchom kilka eksperymentów z różnymi wartościami parametrów i porównaj dokładność wytrenowanego modelu na zbiorze walidacyjnym.
3.   Zbadaj wpływ wybranych hiperparametrów modelu (np. liczba warstw liniowych modelu, rozmiary warstw) i procesu uczenia (np. początkowa wartość stopy uczenia, liczba epok, typ i parametry planisty stopy uczenia, typ i parametry optymalizatora) na skuteczność wytrenowanego modelu. Uruchom kilka eksperymentów z różnymi wartościami hiperparametrów i porównaj dokładność wytrenowanego modelu na zbiorze walidacyjnym. Następnie wykonaj finalną ewaluację najlepszego modelu na zbiorze testowym.


##Przygotowanie środowiska
Upewnij się, że notatnik jest uruchomiony na maszynie z GPU. Jesli GPU nie jest dostępne zmień typ maszyny (Runtime | Change runtime type) i wybierz T4 GPU.

In [8]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


Instalacja dodatkowych bibliotek: datasets (z biblioteki HuggingFace), TorchMetrics i W&B (Weights and Biases) Models.

In [9]:
!pip install -q datasets
!pip install -q torchmetrics
!pip install -q wandb

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Import bibliotek.

In [ ]:
import torch.nn as nn
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import DatasetDict
from sklearn.feature_extraction.text import TfidfVectorizer

print(f"Wersja biblioteki PyTorch: {torch.__version__}")

Wersja biblioteki PyTorch: 2.10.0+cu128


Sprawdzenie dostępności GPU.

In [13]:
print(f"Dostępność GPU: {torch.cuda.is_available()}")
print(f"Typ GPU: {torch.cuda.get_device_name(0)}")

Dostępność GPU: False


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
import wandb

# Logowanie do serwisu Weights&Biases monitorującego przebieg eksperymentów
# wandb.login()

# Rozwiązanie

Podział danych

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

ds = load_dataset("Yelp/yelp_review_full")

Generating test split: 100%|██████████| 50000/50000 [00:00<00:00, 545056.00 examples/s]


In [ ]:
train_val = ds['train'].train_test_split(test_size=0.85, seed=42)
val_test = train_val['test'].train_test_split(test_size=0.9, seed=42)

final_dataset = DatasetDict({
    'train': train_val['train'],
    'validation': val_test['train'],
    'test': val_test['test'].select(range(52250))
})

In [30]:
print(ds['test'][0])
print(ds['train'][0])

{'label': 0, 'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'}
{'label': 4, 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospita

In [17]:
print("train: ", len(final_dataset["train"]))
print("validation: ", len(final_dataset["validation"]))
print("test: ", len(final_dataset["test"]))

train:  97500
validation:  55250
test:  52250


In [34]:
type(final_dataset["train"]['text'][0])

str

In [35]:
vectorizer = TfidfVectorizer()
vectorizer.fit_transform(list(final_dataset["train"]['text']))
final_dataset["train"]['tfidf'] = vectorizer.transform(final_dataset["train"]['text'])
final_dataset["validation"]['tfidf'] = vectorizer.transform(final_dataset["validation"]['text'])
final_dataset["test"]['tfidf'] = vectorizer.transform(final_dataset["test"]['text'])

KeyboardInterrupt: 

In [27]:
class ZuziaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm = nn.BatchNorm1d(96067)
        self.fc1 = nn.Linear(96067, 256)
        self.dropout = nn.Dropout(p=0.3)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 5)

In [ ]:
def calc_accuracy(logits: torch.Tensor, y_true: torch.Tensor) -> float:
    y_pred_labels = torch.argmax(logits, dim=1).float()
    accuracy = (y_pred_labels == y_true).sum().item() / y_true.size(0)
    return accuracy

In [ ]:
def train_network(model: nn.Module, criterion: nn.Module, optimizer: torch.optim.Optimizer,
                  train_loader: DataLoader, val_data: DatasetDict, n_epochs: int = 200):
    model.train()
    for k in range(n_epochs):
        for batch in train_loader:

            x = batch["tfidf"]
            y = batch["label"]
            
            optimizer.zero_grad()
            logits = model(x)
            # logits ma rozmiar (N, 1)
            logits = logits.squeeze(1)
            # logits ma rozmiar (N,)

            # Wyznacz wartość funkcji straty
            loss = criterion(logits, y)

            # Przejście w tył- wyznaczenie gradientu funkcji straty względem parametrów (wag) modelu
            loss.backward()

            # Krok optymalizacji metodą spadku wzdłuż gradientu
            optimizer.step()

        train_accuracy = calc_accuracy(logits, y)

        # Oblicz dokładność klasyfikacji na zbiorze testowym
        with torch.inference_mode():
            model.eval()
            logits = model(val_data["tfidf"])
            model.train()
            test_accuracy = calc_accuracy(logits.squeeze(1), val_data["label"])

        if k % 20 == 0:
            print(f"Epoch: {k}   Wartość funkcji straty: {loss.item():.5f}   Dokładność (train): {train_accuracy:.4f}   Dokładność (test): {test_accuracy:.4f}")

In [ ]:
net = ZuziaNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)

train_loader = DataLoader(final_dataset["train"], batch_size=32, shuffle=True)

train_network(net, criterion, optimizer, train_loader, final_dataset["validation"])

KeyboardInterrupt: 